In [ ]:
import torch
import torchaudio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import scipy.signal
import random
import os
import pickle
import jiwer
from IPython.display import Audio

# Asegurar reproducibilidad
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

## Cargar Tokenizador

In [ ]:
# Cargar tokenizador creado en Tarea 1.1
with open('fechas2_tokenizer_es.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

print(f"Tokenizador cargado. Tamaño del vocabulario: {tokenizer.vocab_size}")
print(f"Tokens especiales: <pad>={tokenizer.word2index['<pad>']}, <sos>={tokenizer.word2index['<sos>']}, <eos>={tokenizer.word2index['<eos>']}")

## Arquitectura del Modelo Transformer

In [ ]:
class FeedForward(torch.nn.Module):
    def __init__(self, d_model=512, d_ff=1024, dropout=0.1, **kwargs):
        super().__init__()
        self.ff = torch.nn.Sequential(
            torch.nn.LayerNorm(d_model),
            torch.nn.Linear(d_model, d_ff),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(d_ff, d_model),
        )
        
    def forward(self, x):
        return self.ff(x)


class SelfAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads=8, d_head=64, dropout=0.1, **kwargs):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_head
        self.scale = torch.sqrt(torch.tensor(d_head, dtype=torch.float32))
        self.norm = torch.nn.LayerNorm(d_model)
        self.q_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.v_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.k_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.dropout = torch.nn.Dropout(dropout)
        self.out = torch.nn.Linear(d_head*n_heads, d_model)

    def forward(self, x):
        x = self.norm(x)
        b = x.shape[0]
        q = self.q_linear(x).view(b, -1, self.n_heads, self.d_head)
        k = self.k_linear(x).view(b, -1, self.n_heads, self.d_head)
        v = self.v_linear(x).view(b, -1, self.n_heads, self.d_head) 

        scores = torch.einsum('bihd,bjhd->bhij', q, k) / self.scale
        att = scores.softmax(dim=-1)
        att = self.dropout(att)

        out = torch.einsum('bhij,bjhd->bihd', att, v).reshape(b, -1, self.n_heads*self.d_head)
        out = self.dropout(out)
        out = self.out(out)
        return out


class Encoder(torch.nn.Module):
    def __init__(self, nb_layers=6, **kwargs):
        super().__init__()        
        self.seq_len = kwargs['seq_len']
        self.pos = torch.nn.Parameter(torch.randn(1, self.seq_len, kwargs['d_model']))
        self.att = torch.nn.ModuleList([SelfAttention(**kwargs) for _ in range(nb_layers)])
        self.ff = torch.nn.ModuleList([FeedForward(**kwargs) for _ in range(nb_layers)])
        
    def forward(self, x):
        b, t, d = x.shape
        x = x + self.pos[:, :t, :]
        for att, ff in zip(self.att, self.ff):
            x = x + att(x)
            x = x + ff(x)            
        return x


class CausalSelfAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads=8, d_head=64, dropout=0.1, **kwargs):
        super().__init__()
        self.seq_len = kwargs['seq_len']
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_head
        self.scale = torch.sqrt(torch.tensor(d_head, dtype=torch.float32))
        self.norm = torch.nn.LayerNorm(d_model)
        self.q_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.v_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.k_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.dropout = torch.nn.Dropout(dropout)
        self.out = torch.nn.Linear(d_head*n_heads, d_model)
        
        self.register_buffer("mask", torch.tril(torch.ones(self.seq_len, self.seq_len))[None, None, ...] == 0)

    def forward(self, x):
        x = self.norm(x)
        b, n, d = x.shape
        q = self.q_linear(x).view(b, -1, self.n_heads, self.d_head)
        k = self.k_linear(x).view(b, -1, self.n_heads, self.d_head)
        v = self.v_linear(x).view(b, -1, self.n_heads, self.d_head) 

        scores = torch.einsum('bihd,bjhd->bhij', q, k) / self.scale
        scores = scores.masked_fill(self.mask[:,:,:n,:n], float('-inf'))
        att = scores.softmax(dim=-1)
        att = self.dropout(att)

        out = torch.einsum('bhij,bjhd->bihd', att, v).reshape(b, -1, self.n_heads*self.d_head)
        out = self.dropout(out)
        out = self.out(out)
        return out


class CrossAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads=8, d_head=64, dropout=0.1, **kwargs):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_head
        self.scale = torch.sqrt(torch.tensor(d_head, dtype=torch.float32))
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.norm2 = torch.nn.LayerNorm(d_model)
        self.q_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.v_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.k_linear = torch.nn.Linear(d_model, d_head*n_heads)
        self.dropout = torch.nn.Dropout(dropout)
        self.out = torch.nn.Linear(d_head*n_heads, d_model)
    
    def forward(self, x1, x2):
        x1 = self.norm1(x1)
        x2 = self.norm2(x2)  
        b = x1.shape[0]
        q = self.q_linear(x1).view(b, -1, self.n_heads, self.d_head)
        k = self.k_linear(x2).view(b, -1, self.n_heads, self.d_head)
        v = self.v_linear(x2).view(b, -1, self.n_heads, self.d_head) 

        scores = torch.einsum('bihd,bjhd->bhij', q, k) / self.scale
        att = scores.softmax(dim=-1)
        att = self.dropout(att)

        out = torch.einsum('bhij,bjhd->bihd', att, v).reshape(b, -1, self.n_heads*self.d_head)
        out = self.dropout(out)
        out = self.out(out)
        return out, att


class Decoder(torch.nn.Module):
    def __init__(self, nb_layers=6, **kwargs):
        super().__init__()        
        self.seq_len = kwargs['seq_len']
        self.pos = torch.nn.Parameter(torch.randn(1, self.seq_len, kwargs['d_model']))
        self.att = torch.nn.ModuleList([CausalSelfAttention(**kwargs) for _ in range(nb_layers)])
        self.cross_att = torch.nn.ModuleList([CrossAttention(**kwargs) for _ in range(nb_layers)])
        self.ff = torch.nn.ModuleList([FeedForward(**kwargs) for _ in range(nb_layers)])
        
    def forward(self, x, enc):
        b, t, d = x.shape
        x = x + self.pos[:, :t, :]
        for att, cross_att, ff in zip(self.att, self.cross_att, self.ff):
            x = x + att(x)
            x = x + cross_att(x, enc)[0]
            x = x + ff(x)            
        return x

## SpecAugment y Feature Extractor

In [ ]:
class SpecAug(torch.nn.Module):
    """SpecAugment: Data augmentation en el dominio del espectrograma."""
    
    def __init__(self, prob_t_warp=0.5,
                       t_factor=(0.9, 1.1), 
                       f_mask_width=(0, 8), 
                       t_mask_width=(0, 10),
                       nb_f_masks=[1,2], 
                       nb_t_masks=[1,2]):
        super().__init__()
        self.t_factor = t_factor
        self.f_mask_width = f_mask_width
        self.t_mask_width = t_mask_width
        self.nb_f_masks = nb_f_masks
        self.nb_t_masks = nb_t_masks
        self.prob_t_warp = prob_t_warp

    def time_warp(self, x):
        x = torch.nn.functional.interpolate(x, size=(int(x.shape[2]*np.random.uniform(*self.t_factor)), ))
        return x
    
    def freq_mask(self, x):
        for _ in range(np.random.randint(*self.nb_f_masks)):
            f = np.random.randint(*self.f_mask_width)
            f0 = np.random.randint(0, x.shape[1]-f)
            x[:,f0:f0+f,:] = 0
        return x

    def time_mask(self, x):
        for _ in range(np.random.randint(*self.nb_t_masks)):
            t = np.random.randint(*self.t_mask_width)
            t0 = np.random.randint(0, x.shape[2]-t)
            x[:,:,t0:t0+t] = 0
        return x

    def forward(self, x):
        if np.random.uniform() < self.prob_t_warp:
            x = self.time_warp(x)
        x = self.freq_mask(x)
        x = self.time_mask(x)
        return x


class AudioFeatures(torch.nn.Module):
    """Extractor de características: Log Mel-Spectrogram."""
    
    def __init__(self, feat_dim=80, d_model=512, **kwargs):
        super().__init__()
        self.fe = torchaudio.transforms.MelSpectrogram(
                        n_fft=512, 
                        win_length=25*16,  # 25ms window
                        hop_length=10*16,  # 10ms shift
                        n_mels=feat_dim)
        self.spec_aug = SpecAug()
        self.linear = torch.nn.Linear(feat_dim, d_model)

    def forward(self, x): 
        x = self.fe(x)        
        x = (x+1e-6).log()
        if self.training:
            x = self.spec_aug(x)
        x = x.transpose(1, 2)
        x = self.linear(x)
        return x

## Modelo Completo

In [ ]:
class AudioTransformer(torch.nn.Module):
    """Transformer encoder-decoder para ASR."""
    
    def __init__(self, vocab_size, **kwargs):
        super().__init__()
        self.vocab_size = vocab_size
        self.seq_len = kwargs['seq_len']

        self.fe = AudioFeatures(**kwargs)
        self.enc = Encoder(**kwargs)
        
        self.emb = torch.nn.Embedding(vocab_size, kwargs['d_model'])
        self.dec = Decoder(**kwargs)
        self.out = torch.nn.Linear(kwargs['d_model'], vocab_size)

    def encoder(self, x):
        x = self.fe(x)
        return self.enc(x)

    def decoder(self, y, enc):
        y = self.emb(y)
        dec = self.dec(y, enc)
        return self.out(dec)
    
    def forward(self, x, y):
        enc = self.encoder(x)
        return self.decoder(y, enc)
                
    def loss(self, x, y):        
        logits = self(x, y[:,:-1])
        target = y[:,1:]
        loss = torch.nn.functional.cross_entropy(
            logits.reshape(-1, self.vocab_size), 
            target.reshape(-1)
        )
        return loss
    
    def generate(self, x, max_len=None):
        """Genera texto usando greedy decoding."""
        device = next(self.parameters()).device
        self.eval()
        
        if max_len is None:
            max_len = self.seq_len
        
        # Empezar con token <sos>
        y = [tokenizer.word2index['<sos>']]
        
        with torch.no_grad():            
            enc = self.encoder(x.to(device))   
            
            while y[-1] != tokenizer.word2index['<eos>'] and len(y) < max_len:
                logits = self.decoder(torch.tensor(y).unsqueeze(0).to(device), enc)
                y.append(logits.argmax(-1)[:,-1].item())
                            
        return y

## Dataset con Data Augmentation

In [ ]:
class NoiseAug(object):
    """Augmentation con ruido aditivo."""
    
    def __init__(self, noise_dir='musan_small/', prob=0.5):
        self.prob = prob
        self.noises = glob.glob(noise_dir+'/**/*.wav', recursive=True)
        print(f"Noise files loaded: {len(self.noises)}")
        
    def __call__(self, x):
        if np.random.uniform() < self.prob:
            if len(self.noises) == 0:
                return x
            n = torchaudio.load(np.random.choice(self.noises))[0][0]            
            if len(n) < len(x):
                n = torch.nn.functional.pad(n, (0, len(x)-len(n)), value=0)
            elif len(n) > len(x):
                t0 = np.random.randint(0, len(n) - len(x))
                n = n[t0:t0+len(x)]
            n = n.numpy()
            p_x = x.std()**2
            p_n = n.std()**2
            snr = np.random.uniform(5, 15)
            n = n * np.sqrt(p_x/p_n) * np.power(10, -snr/20)
            x = x + n
        return x


class RIRAug(object):
    """Augmentation con reverberación (RIR)."""
    
    def __init__(self, rir_dir='RIRS_NOISES_small/', prob=0.5):
        self.prob = prob
        self.rirs = glob.glob(rir_dir+'/**/*.wav', recursive=True)
        print(f"RIR files loaded: {len(self.rirs)}")

    def __call__(self, x):
        if np.random.uniform() < self.prob:
            if len(self.rirs) == 0:
                return x
            n = len(x)
            rir = torchaudio.load(np.random.choice(self.rirs))[0][0]
            rir = rir.numpy()
            rir = rir / np.max(np.abs(rir))
            x = scipy.signal.convolve(x, rir)
            t0 = np.argmax(np.abs(rir))
            x = x[t0:t0+n]
        return x


class Fechas2ASRDataset(torch.utils.data.Dataset):
    """Dataset para ASR de fechas con augmentation."""
    
    def __init__(self, csv_file, tokenizer, audio_len=4*16000, transform=[], max_text_len=20):
        self.df = pd.read_csv(csv_file)
        self.tokenizer = tokenizer
        self.transform = transform
        self.audio_len = audio_len
        self.max_text_len = max_text_len
        self.csv_dir = os.path.dirname(csv_file)
        
        print(f"Dataset loaded: {len(self.df)} examples")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Cargar audio
        audio_path = os.path.join(self.csv_dir, row['wav'])
        if not os.path.exists(audio_path):
            audio_path = os.path.join('..', row['wav'])
        
        x, fs = torchaudio.load(audio_path)
        
        # Padding/truncate
        if x.shape[1] < self.audio_len:
            x = torch.nn.functional.pad(x, (0, self.audio_len-x.shape[1]), value=0)
        else:
            x = x[:, :self.audio_len]

        x = x[0].numpy()
        
        # Aplicar augmentation
        for t in self.transform:
            x = t(x)

        # Tokenizar texto
        text = row['txt']
        y = self.tokenizer.encode(text, seq_len=self.max_text_len)
        
        return torch.tensor(x, dtype=torch.float32), y


class Fechas2TestDataset(torch.utils.data.Dataset):
    """Dataset de test sin augmentation."""
    
    def __init__(self, csv_file, tokenizer, audio_len=4*16000, max_text_len=20):
        self.df = pd.read_csv(csv_file)
        self.tokenizer = tokenizer
        self.audio_len = audio_len
        self.max_text_len = max_text_len
        self.csv_dir = os.path.dirname(csv_file)
        
        print(f"Test dataset loaded: {len(self.df)} examples")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        audio_path = os.path.join(self.csv_dir, row['wav'])
        if not os.path.exists(audio_path):
            audio_path = os.path.join('..', row['wav'])
        
        x, fs = torchaudio.load(audio_path)
        
        if x.shape[1] < self.audio_len:
            x = torch.nn.functional.pad(x, (0, self.audio_len-x.shape[1]), value=0)
        else:
            x = x[:, :self.audio_len]

        x = x[0]
        
        text = row['txt']
        y = self.tokenizer.encode(text, seq_len=self.max_text_len)
        
        return x, y, text  # Devolver también el texto original


# Crear datasets
trainset = Fechas2ASRDataset(
    '../fechas2/fechas2_train.es.csv',
    tokenizer,
    transform=[NoiseAug(prob=0.5), RIRAug(prob=0.5)]
)

testset = Fechas2TestDataset(
    '../fechas2/fechas2_test.es.csv',
    tokenizer
)

## Entrenar el Modelo

In [ ]:
# Configuración del modelo
model_config = {
    'vocab_size': tokenizer.vocab_size,
    'd_model': 256,
    'nb_layers': 6,
    'd_ff': 512,
    'n_heads': 8,
    'd_head': 32,
    'dropout': 0.1,
    'seq_len': 500,
    'feat_dim': 80
}

model = AudioTransformer(**model_config)

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Configurar device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
model.to(device)

# Optimizador
opt = torch.optim.Adam(model.parameters(), lr=3e-4)

# Configuración de entrenamiento
nb_epochs = 10  # Ajustar según necesidad
batch_size = 16  # Ajustar según GPU disponible

# DataLoader
trainloader = torch.utils.data.DataLoader(
    trainset, 
    batch_size=batch_size, 
    shuffle=True,
    num_workers=0  # Ajustar según sistema
)

print(f"\nStarting training for {nb_epochs} epochs...")
print(f"Batches per epoch: {len(trainloader)}")

In [ ]:
# Entrenamiento
model.train()
losses = []

for e in range(nb_epochs):
    epoch_loss = 0
    for batch_idx, (x, y) in enumerate(trainloader):
        x = x.to(device)
        y = y.to(device)
        
        opt.zero_grad()
        loss = model.loss(x, y)
        loss.backward()
        opt.step()
        
        epoch_loss += loss.item()
        
        # Mostrar progreso cada 100 batches
        if (batch_idx + 1) % 100 == 0:
            print(f'  Batch {batch_idx+1}/{len(trainloader)}: loss={loss.item():.4f}')
    
    avg_loss = epoch_loss / len(trainloader)
    losses.append(avg_loss)
    print(f'Epoch {e+1}/{nb_epochs}: avg_loss={avg_loss:.4f}')

# Guardar modelo
torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'config': model_config}, 
           'model_fechas2_es.pt')
print("\nModelo guardado en 'model_fechas2_es.pt'")

## Visualizar Curva de Aprendizaje

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.xlabel('Época')
plt.ylabel('Loss promedio')
plt.title('Curva de Aprendizaje')
plt.grid(True)
plt.show()

## Evaluar el Modelo (WER)

In [ ]:
# Cargar modelo si es necesario
# checkpoint = torch.load('model_fechas2_es.pt')
# model.load_state_dict(checkpoint['model'])

model.eval()
hyp_ = []
ref_ = []

print("Evaluando modelo en test set...")
for i, (x, y, text_orig) in enumerate(testset):    
    x = x.to(device)    
    y_pred = model.generate(x[None,...])

    # Decodificar predicción
    hyp = tokenizer.decode(y_pred)
    
    # Referencia original
    ref = text_orig
    
    hyp_.append(hyp)
    ref_.append(ref)
    
    # Mostrar algunos ejemplos
    if i < 10:
        print(f"\nEjemplo {i+1}:")
        print(f"  REF: {ref}")
        print(f"  HYP: {hyp}")

# Calcular WER
out = jiwer.process_words(ref_, hyp_)
print(f"\n{'='*50}")
print(f"WER (Word Error Rate): {out.wer:.2%}")
print(f"Substituciones: {out.substitutions}")
print(f"Deleciones: {out.deletions}")
print(f"Inserciones: {out.insertions}")
print(f"{'='*50}")

## Visualizar Atención

In [ ]:
# Añadir hooks para capturar atención
att_ = [None] * len(model.dec.cross_att)
for i, cross_att in enumerate(model.dec.cross_att):
    cross_att.register_forward_hook(
        lambda _, ins, outs, index=i: att_.__setitem__(index, outs[-1])
    )

os.makedirs('out', exist_ok=True)
print("Hooks de atención registrados")

In [ ]:
def visualize_example(idx, testset, model, device, tokenizer):
    """Visualiza ejemplo con atención y espectrograma."""
    x, y, text = testset[idx]
    
    x = x.to(device)
    y = y.to(device)
    
    # Generar predicción
    y_pred = model.generate(x[None,...])
    
    # Forzar forward para capturar atención
    _ = model(x[None,:], y[None,:]) 
    
    # Decodificar
    hyp = tokenizer.decode(y_pred)
    ref = text
    
    print(f"\nEjemplo {idx}:")
    print(f"  REF: {ref}")
    print(f"  HYP: {hyp}")
    
    # Calcular atención promedio
    mean_att = torch.cat(att_).mean([0, 1])
    
    # Obtener tokens
    y_tokens = [tokenizer.index2word.get(i, '?') for i in y.cpu().numpy().tolist()]
    
    # Visualizar
    plt.figure(figsize=(15, 8))
    
    # Atención
    plt.subplot(2,1,1)
    plt.imshow(mean_att.cpu().detach().numpy(), interpolation='none', cmap='jet', aspect='auto')
    ax = plt.gca()
    ax.set_yticks(np.arange(len(y_tokens)), y_tokens)
    plt.title('Attention Map')
    plt.xlabel('Audio frames')
    plt.ylabel('Output tokens')
    plt.colorbar()
    
    # Espectrograma
    plt.subplot(2,1,2)
    spec = model.fe.fe(x).log().detach().cpu().numpy()
    plt.imshow(spec, interpolation='none', cmap='jet', aspect='auto', origin='lower')
    plt.title('Log Mel Spectrogram')
    plt.xlabel('Frames')
    plt.ylabel('Mel bins')
    plt.colorbar()
    
    plt.tight_layout()
    plt.show()
    
    # Reproducir audio
    audio_path = testset.df.iloc[idx]['wav']
    if not os.path.exists(audio_path):
        audio_path = os.path.join('..', audio_path)
    return Audio(audio_path)

# Visualizar varios ejemplos
for idx in [0, 10, 50, 100]:
    audio = visualize_example(idx, testset, model, device, tokenizer)
    display(audio)

## Análisis de Errores

In [ ]:
# Encontrar ejemplos con errores
errors = []
for i, (ref, hyp) in enumerate(zip(ref_, hyp_)):
    if ref != hyp:
        errors.append((i, ref, hyp))

print(f"\nTotal de errores: {len(errors)} / {len(ref_)} ({len(errors)/len(ref_)*100:.1f}%)")
print(f"\nPrimeros 10 errores:")
for i, (idx, ref, hyp) in enumerate(errors[:10]):
    print(f"\n{i+1}. Índice {idx}:")
    print(f"   REF: {ref}")
    print(f"   HYP: {hyp}")

## Guardar Resultados

In [ ]:
# Guardar predicciones
results_df = pd.DataFrame({
    'reference': ref_,
    'hypothesis': hyp_
})
results_df.to_csv('results_fechas2_es.csv', index=False)
print("Resultados guardados en 'results_fechas2_es.csv'")

# Guardar métricas
metrics = {
    'WER': out.wer,
    'substitutions': out.substitutions,
    'deletions': out.deletions,
    'insertions': out.insertions,
    'total_words': len(' '.join(ref_).split()),
    'num_examples': len(ref_)
}

with open('metrics_fechas2_es.pkl', 'wb') as f:
    pickle.dump(metrics, f)

print("Métricas guardadas en 'metrics_fechas2_es.pkl'")